# Philadelphia, PA — Land Value Tax Model

**Policy assumptions:**
- **Levy scope**: Full combined rate (city + school district) — 1.3998% = 13.998 mills applied to 100% of assessed value
- **Reform**: Split-rate 4:1 (land taxed at 4× the improvement millage rate)
- **Exemptions**: Preserve all existing exemptions — homestead (~$100K assessed-value deduction for owner-occupants), active construction abatements, and institutional/nonprofit exemptions are already embedded in OPA's `taxable_land` and `taxable_building` columns
- **Data source**: Philadelphia Office of Property Assessment (OPA) via Carto public API
- **Revenue target**: ~$796M (FY2024 current-year real estate tax collections)

**Key limitation**: OPA's land/building split defaults to 20% land for ~45% of improved parcels
(especially multi-family and commercial). This understates land value and attenuates the split-rate
impact relative to a market-derived land assessment.

In [1]:
import os
import sys
import io
import urllib.parse
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from lvt.philadelphia import tax_year_params, parcel_cache_path, split_zero_building_parcels
from lvt.policy_analysis import analyze_vacant_land, print_vacant_land_summary

CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
LAND_IMPROVEMENT_RATIO = 4.0

# --- Tax year -------------------------------------------------------------------
# Rates and the revenue-validation target live in lvt/philadelphia.py, keyed by year and
# cited there. Do NOT hardcode a millage here: the combined rate has been 1.3998% for
# years, but the City/School split moved at TY2025, which silently invalidates the
# city-only cross-check without changing anything the model computes.
TAX_YEAR = int(os.environ.get('LVT_TAX_YEAR', 2026))   # override: LVT_TAX_YEAR=2027
TY = tax_year_params(TAX_YEAR)
MILLAGE = TY.combined_mills
PARCEL_PATH = parcel_cache_path(TAX_YEAR)
MODEL_TYPE = f'split_rate_4to1_ty{TAX_YEAR}'
EXPORT_SUFFIX = f'_ty{TAX_YEAR}'

print(TY.describe())

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

C:\Users\druss\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


TY2026: 0.6159% city + 0.7839% school = 1.3998% (13.998 mills) | city target $891,102,000 (projection) | homestead $100,000


## Section 2 — Fetch / Load Parcel Data

**Column mapping**

| Concept | Source | Column | Notes |
|---|---|---|---|
| Land value (taxable) | `assessments` year=2024 | `taxable_land` | Post-exemption; 2024 vintage matches FY2024 billing |
| Improvement value (taxable) | `assessments` year=2024 | `taxable_building` | Post-exemption; 2024 vintage |
| Total assessed value | `assessments` year=2024 | `market_value` | Pre-exemption gross |
| Use/class code | `opa_properties_public` | `category_code` | Integer 1–15 |
| Owner name(s) | `opa_properties_public` | `owner_1`, `owner_2` | Coalesced into `owner_name` in Section 3; used by the Section 6 ownership-concentration analysis |
| Geometry | `opa_properties_public` | `the_geom` | WKB point (parcel centroid) |
| PIN (DOR key) | `opa_properties_public` | `pin` | Join key for the LYCD notebooks' lot-area chain |
| Lot area (sq ft) | `opa_properties_public` | `total_area` | Used by the LYCD notebooks' lot-area chain |

**Why 2024 assessments?**
The current OPA table reflects 2026 reassessments. FY2024 taxes were billed on 2024 assessments.
Using 2026 values overstates the modeled city levy by ~21% vs. actual FY2024 collections.
The Carto `assessments` history table has per-year taxable values; using `year=2024` brings
the city-levy cross-check to within ~5% of actual FY2024 collections. The residual ~5% gap
is current-year delinquency (non-collection), which is ~4–5% historically for Philadelphia.

**Assessment ratio**: 100% full market value (no assessment ratio)

**Millage source**: City of Philadelphia combined rate: 1.3998% = 13.998 mills
(city levy 0.6317% + school district 0.7681%)

In [2]:
if not PARCEL_PATH.exists():
    raise FileNotFoundError(
        f'{PARCEL_PATH} not found. Build it with:\n'
        f'    python scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR}\n'
        'The cache is keyed by tax year on purpose — opa_properties_public always carries '
        'the latest assessment year, so an unsuffixed cache makes it easy to model one '
        "year's taxable values against another year's expectations with no visible symptom."
    )
gdf = gpd.read_parquet(PARCEL_PATH)
_required = {'parcel_number', 'taxable_land', 'taxable_building', 'market_value',
             'exempt_land', 'exempt_building', 'pin', 'category_code', 'total_area'}
_missing = _required - set(gdf.columns)
if _missing:
    raise ValueError(
        f'{PARCEL_PATH} is missing columns {sorted(_missing)} — rebuild with '
        f'scripts/build_philadelphia_parcel_cache.py --year {TAX_YEAR} --force'
    )
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
print(f'Loaded {len(gdf):,} parcels for TY{TAX_YEAR}')
print(f'  taxable base: ${(gdf["taxable_land"].sum() + gdf["taxable_building"].sum())/1e9:.3f}B')


Loaded 583,249 parcels for TY2026
  taxable base: $152.997B


## Section 3 — Classify and Validate

In [3]:
# Owner name: coalesce owner_1 -> owner_2 (used by the Section 6 ownership-concentration analysis)
gdf['owner_name'] = gdf[['owner_1', 'owner_2']].replace('', np.nan).bfill(axis=1).iloc[:, 0].fillna('')

# Coerce category_code to string for mapping
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement → Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: a $0 taxable building line has three different causes, and calling all of
# them "abated" put ~13K homesteaded rowhomes in the abated bucket -- then revoked their
# Homestead Exemption under the reform. Split on the year's statutory homestead cap.
GENUINE_VACANT_CODES = {'6', '12', '13'}
_zb = split_zero_building_parcels(
    gdf, gdf['PROPERTY_CATEGORY'], TY.homestead_exemption, CATEGORY_MAP,
    genuine_vacant_codes=tuple(GENUINE_VACANT_CODES),
)
gdf['PROPERTY_CATEGORY'] = _zb.category
abated_mask = _zb.abated
print(_zb.describe())

# Override 3 (NEW): OPA classifies ~1,483 parcels as vacant (codes 6/12/13) but records
# a non-zero taxable_building — small structures on lots that were never recategorized.
# Their improvement value makes them behave like improved parcels, not vacant land.
# Move them to "Improved Vacant Land" so they don't distort the Vacant Land result.
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

# Full exemption flag: both taxable values are zero
gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: re-classify fully exempt parcels by their OPA type with "— Exempt" suffix.
EXEMPT_CATEGORY_MAP = {k: v + ' — Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other — Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

zero-building line: 14,287 abated | 13,995 homestead-zeroed (96.3% confirmed by OPA's homestead flag) | 1,119 genuinely $0 improvement
Total parcels: 583,249
Fully exempt: 36,932  |  Abated: 14,287  |  Improved vacant: 880  |  Taxable: 546,317

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential                  430570
Small Multi-Family (2-4 units)              38685
Vacant Land                                 30557
Single Family Residential — Exempt          20078
Abated / Construction Exemption             14287
Mixed Use                                   13743
Vacant Land — Exempt                        11714
Commercial                                   8802
Industrial                                   3553
Commercial — Exempt                          3298
Large Multi-Family (5+ units)                3002
Other Residential                            1108
Small Multi-Family (2-4 units) — Exempt      1026
Improved Vacant Land                          880
Offic

## Section 4 — Current Tax Model

Philadelphia's combined real estate tax rate is **1.3998% = 13.998 mills** applied to 100% of assessed value.

The 2024 assessment-year `taxable_land` and `taxable_building` values embed all existing exemptions
as of the 2024 billing cycle:
- **Homestead Exemption** (~$100K assessed-value deduction): reduces `taxable_building`
- **10-year construction abatement**: zeroes `taxable_building` for abated parcels
- **Institutional / nonprofit**: both columns are $0 for fully exempt parcels

Current tax = `(taxable_land + taxable_building) × 13.998 / 1000`

Cross-check: city-only portion at 0.6317% should be within ~5% of FY2024 city actuals (~$796M).
Residual gap is current-year delinquency/non-collection, not a modeling error.

In [4]:
gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

city_revenue = gdf['taxable_total'].mul(TY.city_mills / 1000).sum()
print(f'Modeled combined levy (city + school):  ${current_revenue:,.0f}')
print(f'Implied city-only portion ({TY.city_rate_pct}%):   ${city_revenue:,.0f}')

if TY.city_revenue_target is None:
    print(f'\nNO REVENUE VALIDATION for TY{TAX_YEAR}.')
    print(f'  {TY.source}')
else:
    gap_pct = (city_revenue / TY.city_revenue_target - 1) * 100
    print(f'City-only target ({TY.target_kind}):            ${TY.city_revenue_target:,}')
    print(f'City portion gap: {gap_pct:+.2f}%  (expected: a few % over, from delinquency)')
    assert abs(gap_pct) < 10.0, (
        f'City gap {gap_pct:.2f}% exceeds 10% for TY{TAX_YEAR}. Check that the assessment '
        f'year, the City rate ({TY.city_rate_pct}%) and the revenue target all refer to the '
        'same billing year — see lvt/philadelphia.py.'
    )


Modeled combined levy (city + school):  $2,141,653,043
Implied city-only portion (0.6159%):   $942,308,979
City-only target (projection):            $891,102,000
City portion gap: +5.75%  (expected: a few % over, from delinquency)


## Section 5 — Split-Rate Model (4:1)

In [5]:
# For abated parcels, use OPA's own recorded assessed building value (`exempt_building`),
# with a market_value - taxable_land fallback for the ~2,100 mid-construction parcels where
# exempt_building = 0 (same approach as model_post_abatement.ipynb). Current_tax stays as
# actual (land-only under abatement) -- only the reform-scenario building base changes.
gdf['model_land'] = pd.to_numeric(gdf['taxable_land'], errors='coerce').fillna(0).clip(lower=0)
gdf['model_building'] = pd.to_numeric(gdf['taxable_building'], errors='coerce').fillna(0).clip(lower=0)

abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'
exempt_bldg  = pd.to_numeric(gdf['exempt_building'], errors='coerce').fillna(0)
market_val   = pd.to_numeric(gdf['market_value'],    errors='coerce').fillna(0)
tax_land     = pd.to_numeric(gdf['taxable_land'],     errors='coerce').fillna(0)
implied_bldg = (market_val - tax_land).clip(lower=0)
abated_bldg  = exempt_bldg.where(exempt_bldg > 0, implied_bldg)
gdf.loc[abated, 'model_building'] = abated_bldg[abated].values

n_exempt_bldg = int((abated & (exempt_bldg > 0)).sum())
n_fallback    = int((abated & (exempt_bldg <= 0)).sum())
abated_bldg_total = gdf.loc[abated, 'model_building'].sum()
print(f'Abated parcels using exempt_building:        {n_exempt_bldg:,}')
print(f'Abated parcels using market_value fallback:  {n_fallback:,}')
print(f'Total abated building base added to reform:  ${abated_bldg_total/1e9:.2f}B')
print()

# Exclude fully-exempt parcels from the reform
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels (unchanged)
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia — 4:1 Split-Rate Tax Impact')


Abated parcels using exempt_building:        14,269
Abated parcels using market_value fallback:  18
Total abated building base added to reform:  $16.21B



Land millage:        28.7268 mills
Improvement millage: 7.1817 mills
Revenue check:       $2,141,653,043 (target: $2,141,653,043)




Philadelphia — 4:1 Split-Rate Tax Impact
                               Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
              Single Family Residential 430570   $-136,434,797      -11.0%      $-317        $-236   -0.6%     -12.1%            20.8%            52.3%
         Small Multi-Family (2-4 units)  38685    $-30,723,030      -12.2%      $-794        $-686  -12.5%     -17.9%             5.2%            80.2%
                            Vacant Land  30557     $55,011,509      105.2%     $1,800         $502  105.2%     105.2%           100.0%             0.0%
     Single Family Residential — Exempt  20078              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
        Abated / Construction Exemption  14287    $155,156,179      421.7%    $10,860       $4,601  383.4%     310.4%           100.0%             0.0%
                              Mixed Use  13743

## Validation Summary

| Check | Result |
|---|---|
| Revenue match | City-only cross-check within ~5% of FY2024 actuals using 2024 assessments (see Section 4) |
| Assessment vintage | 2024 (matches FY2024 billing); current OPA has 2026 values (+16% higher) |
| Residual gap | ~5% — current-year delinquency/non-collection (typical for Philadelphia) |
| Exemption handling | Homestead, abatement, institutional exemptions embedded in 2024 OPA taxable values |
| Land/improvement split | OPA official values; ~45% of improved parcels default to 20% land fraction |
| Millage rate | 13.998 mills (city 0.6317% + school 0.7681% = 1.3998%) |

## Section 6 — Ownership Concentration (Vacant Land)

Groups vacant-land parcels (`PROPERTY_CATEGORY == 'Vacant Land'`) by owner name and reports what
share of vacant land value is held by the top 5%/10% of owners, via `analyze_vacant_land()` from
`lvt.policy_analysis` — the same function already run for Baltimore. Fully exempt parcels are
excluded (`exemption_flag_col='full_exmp'`); "Improved Vacant Land" and the "— Exempt" vacant
bucket are intentionally out of scope, matching the Baltimore call's scoping.

**Limitation**: this measures concentration among distinct OPA `owner_1`/`owner_2` name strings
only. It does not resolve LLC/shell-company networks that share a beneficial owner under different
registered names, and it does not use mailing-address grouping — so any concentration figure below
is a **lower bound** on true ownership concentration.

In [6]:
vacant_results = analyze_vacant_land(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    property_type_col='PROPERTY_CATEGORY',
    vacant_identifier='Vacant Land',
    owner_col='owner_name',
    exemption_flag_col='full_exmp',
)
print_vacant_land_summary(vacant_results)

if 'top_10_owners_by_value' in vacant_results:
    print('\nTop 10 vacant-land owners by adjusted land value:')
    print(vacant_results['top_10_owners_by_value'].to_string())

VACANT LAND ANALYSIS SUMMARY
Total vacant parcels: 30,557
Total vacant land value: $3,734,956,578
Average vacant land value: $122,229
Vacant land as % of total city land value: 8.7%

Ownership concentration:
Top 5% of owners control: $2,335,530,546 (62.5%)
Top 10% of owners control: $2,690,973,905 (72.0%)

Top 10 vacant-land owners by adjusted land value:
                           parcel_count  total_land_value
owner_name                                               
CONRAIL                              43        43536700.0
PAVILION EAST ASSOC                   1        37688400.0
CENTRA ASSOCIATES LP                 23        37671300.0
ACADEMIC PROPERTIES INC               1        35735000.0
3 LOGAN LLC                           1        27588000.0
PIDC/DEVELOPMENT CORP                 1        24868800.0
THE TRUSTEES UNIVERSITY               1        24000000.0
LIBERTY PROPERTY 19TH & A             5        23835700.0
DREXEL UNIVERSITY                     3        22440400.0
UNIV

In [7]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [8]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}{EXPORT_SUFFIX}',
    output_path=f'../../analysis/data/{CITY_NAME}{EXPORT_SUFFIX}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    parcel_id_col='parcel_number',
)

# Standard report — 7 PNGs in analysis/reports/philadelphia/
from lvt.viz import create_city_report
CATEGORY_FOOTNOTE = (
    "Abated / Construction Exemption parcels currently pay tax on land only (taxable_building = $0 under the "
    "10-year abatement). Under the reform their building value uses OPA's own recorded assessed value "
    "(exempt_building, with a market_value - taxable_land fallback where that field is $0), the same "
    "observed value used as the post-abatement baseline in model_post_abatement.ipynb."
)

create_city_report(out_df, f'{CITY_NAME}{EXPORT_SUFFIX}', show=False,
                   category_chart_footnote=CATEGORY_FOOTNOTE)
print('Done.')

  [warn] philadelphia_ty2026: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial — Exempt', 'Hotel — Exempt', 'Improved Vacant Land', 'Industrial — Exempt', 'Large Multi-Family (5+ units) — Exempt', 'Mixed Use — Exempt', 'Office / Commercial Condo — Exempt', 'Other Commercial — Exempt', 'Other Residential — Exempt', 'Other — Exempt', 'Retail / General Commercial — Exempt', 'Single Family Residential — Exempt', 'Small Multi-Family (2-4 units) — Exempt', 'Vacant Land — Exempt']


  ✓ philadelphia_ty2026: 583,249 rows → ../../analysis/data/philadelphia_ty2026.csv  [model: split_rate_4to1_ty2026]


create_city_report: excluded 36,932 fully-exempt parcels (583,249 → 546,317 modeled).


Done.
